In [3]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_classic.retrievers import MultiQueryRetriever


In [4]:
# Relevant health & wellness documents
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [5]:
embedding_model=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

vectorstore=FAISS.from_documents(documents=all_docs,embedding=embedding_model)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9576.89it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
similarity_retriever=vectorstore.as_retriever(search_type='similarity',search_kwargs={'k':5})

In [9]:
llm=HuggingFaceEndpoint(
    repo_id="deepseek-ai/DeepSeek-V3",
    task="text-generation"
)

model=ChatHuggingFace(llm=llm)

multiquery_retriever=MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={'k':5}),
    llm=model
)

In [10]:
query='how to improve energy levels and maintain balance?'

In [11]:
similarity_result=similarity_retriever.invoke(query)
multiquery_result=multiquery_retriever.invoke(query)

In [12]:
for i,doc in enumerate(similarity_result):
    print(f'\n---  Result {i+1}---')
    print(doc.page_content)

print("*"*150)

for i,doc in enumerate(multiquery_result):
    print(f'\n---  Result {i+1}---')
    print(doc.page_content)



---  Result 1---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

---  Result 2---
The solar energy system in modern homes helps balance electricity demand.

---  Result 3---
Consuming leafy greens and fruits helps detox the body and improve longevity.

---  Result 4---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

---  Result 5---
Photosynthesis enables plants to produce energy by converting sunlight.
******************************************************************************************************************************************************

---  Result 1---
Black holes bend spacetime and store immense gravitational energy.

---  Result 2---
Python balances readability with power, making it a popular system design language.

---  Result 3---
Photosynthesis enables plants to produce energy by converting sunlight.

---  Result 4---
Consuming leafy greens and fruits helps detox the body and improve longevity.